# **Meeple Analytics**

**1. Objetivo del proyecto**

**Cliente**

Editorial internacional de juegos de mesa.

**Problema**

La empresa quiere entender qué factores influyen en el éxito de un juego para orientar futuras inversiones y lanzamientos.

**1- Revisar nulos:**

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Cargar dataset
df = pd.read_csv("games.csv")

# Información general
print(df.shape)

(21925, 48)


In [4]:
df.head()

,BGGId,Name,Description,YearPublished,GameWeight,AvgRating,BayesAvgRating,StdDev,MinPlayers,MaxPlayers,...,Rank:partygames,Rank:childrensgames,Cat:Thematic,Cat:Strategy,Cat:War,Cat:Family,Cat:CGS,Cat:Abstract,Cat:Party,Cat:Childrens
0,1,Die Macher,die macher game seven sequential political rac...,1986,4.3206,7.61428,7.10363,1.57979,3,5,...,21926,21926,0,1,0,0,0,0,0,0
1,2,Dragonmaster,dragonmaster tricktaking card game base old ga...,1981,1.9630,6.64537,5.78447,1.45440,3,4,...,21926,21926,0,1,0,0,0,0,0,0
2,3,Samurai,samurai set medieval japan player compete gain...,1998,2.4859,7.45601,7.23994,1.18227,2,4,...,21926,21926,0,1,0,0,0,0,0,0
3,4,Tal der Könige,triangular box luxurious large block tal der k...,1992,2.6667,6.60006,5.67954,1.23129,2,4,...,21926,21926,0,0,0,0,0,0,0,0
4,5,Acquire,acquire player strategically invest business t...,1964,2.5031,7.33861,7.14189,1.33583,2,6,...,21926,21926,0,1,0,0,0,0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21925 entries, 0 to 21924
Data columns (total 48 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BGGId                21925 non-null  int64  
 1   Name                 21925 non-null  object 
 2   Description          21924 non-null  object 
 3   YearPublished        21925 non-null  int64  
 4   GameWeight           21925 non-null  float64
 5   AvgRating            21925 non-null  float64
 6   BayesAvgRating       21925 non-null  float64
 7   StdDev               21925 non-null  float64
 8   MinPlayers           21925 non-null  int64  
 9   MaxPlayers           21925 non-null  int64  
 10  ComAgeRec            16395 non-null  float64
 11  LanguageEase         16034 non-null  float64
 12  BestPlayers          21925 non-null  int64  
 13  GoodPlayers          21925 non-null  object 
 14  NumOwned             21925 non-null  int64  
 15  NumWant              21925 non-null 

In [6]:
df.isnull().sum().sort_values(ascending=False)

,0
Family,15262
LanguageEase,5891
ComAgeRec,5530
ImagePath,17
Description,1
AvgRating,0
GameWeight,0
BGGId,0
StdDev,0
BayesAvgRating,0


In [7]:
df.duplicated().sum()

np.int64(0)

Tras ver los nulos y duplicados y el aporte que esas columnas hacen al analisis, se decide eliminar:

**Description:** por ser una descripcion de texto libre de los juegos
**Imagepath:** por tratarse solo de la ruta de la imagen del juego que no aporta análisis estadístico ni visualizaciones de negocio.

en cambio, a pesar de su cantidad de nulos no eliminamos las siguientes:

**Family:** Tiene muchos nulos porque muchos juegos no pertenecen a una franquicia. Pero podría permitir preguntas como:

*¿Las franquicias venden mejor que los juegos independientes?*

**LanguageEase:** Esta variable es bastante rara y muy interesante.

Mide el nivel de dependencia del idioma de un juego, lo necesario que es para jugarlo.

Ejemplo:

Ajedrez → muy fácil lingüísticamente
Sherlock Holmes Consulting Detective → muy dependiente del idioma

Pregunta de negocio:

*¿Los juegos más accesibles internacionalmente reciben mejores valoraciones?*

Eso podría dar una conclusión muy potente para una editorial.

**ComAgeRec vs MfgAgeRec**

Tiene dos variables:

*ComAgeRec*

Edad recomendada por la comunidad.

*MfgAgeRec*

Edad recomendada por el fabricante.

Esto permite responder una pregunta muy interesante:

¿Los jugadores consideran que los juegos son adecuados para edades diferentes a las recomendadas por la empresa?
*texto en cursiva*
Quiza necesitemos responder a estas preguntas más adelante.

In [8]:
df = df.drop(
    columns=["Description", "ImagePath"],
    errors="ignore"
)

df.columns.tolist()

['BGGId',
 'Name',
 'YearPublished',
 'GameWeight',
 'AvgRating',
 'BayesAvgRating',
 'StdDev',
 'MinPlayers',
 'MaxPlayers',
 'ComAgeRec',
 'LanguageEase',
 'BestPlayers',
 'GoodPlayers',
 'NumOwned',
 'NumWant',
 'NumWish',
 'NumWeightVotes',
 'MfgPlaytime',
 'ComMinPlaytime',
 'ComMaxPlaytime',
 'MfgAgeRec',
 'NumUserRatings',
 'NumComments',
 'NumAlternates',
 'NumExpansions',
 'NumImplementations',
 'IsReimplementation',
 'Family',
 'Kickstarted',
 'Rank:boardgame',
 'Rank:strategygames',
 'Rank:abstracts',
 'Rank:familygames',
 'Rank:thematic',
 'Rank:cgs',
 'Rank:wargames',
 'Rank:partygames',
 'Rank:childrensgames',
 'Cat:Thematic',
 'Cat:Strategy',
 'Cat:War',
 'Cat:Family',
 'Cat:CGS',
 'Cat:Abstract',
 'Cat:Party',
 'Cat:Childrens']

In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
BGGId,21925.0,117652.663216,104628.721777,1.000000,12346.000000,105305.00000,206169.00000,349161.00000
YearPublished,21925.0,1985.494914,212.486214,-3500.000000,2001.000000,2011.00000,2017.00000,2021.00000
GameWeight,21925.0,1.982131,0.848983,0.000000,1.333300,1.96880,2.52520,5.00000
AvgRating,21925.0,6.424922,0.932477,1.041330,5.836960,6.45395,7.05245,9.91429
BayesAvgRating,21925.0,5.685673,0.365311,3.574810,5.510300,5.54654,5.67989,8.51488
StdDev,21925.0,1.516374,0.285578,0.196023,1.320720,1.47688,1.66547,4.27728
MinPlayers,21925.0,2.007343,0.693093,0.000000,2.000000,2.00000,2.00000,10.00000
MaxPlayers,21925.0,5.707868,15.014643,0.000000,4.000000,4.00000,6.00000,999.00000
ComAgeRec,16395.0,10.004391,3.269157,2.000000,8.000000,10.00000,12.00000,21.00000
LanguageEase,16034.0,216.461819,236.595136,1.000000,24.027778,138.00000,351.00000,1757.00000


Hay juegos tradicionales y clásicos como ajedrez o mancala, queremos saber en qué juegos de nueva creacion invertir basándonos en el auge actual de los juegos modernos, asi que se vana a eliminar todos los juegos anteriores a 1950

In [10]:
df_clean = df[df["YearPublished"] >= 1950].copy()

In [11]:
df.nlargest(10, "MaxPlayers")[["Name", "MaxPlayers"]]

,Name,MaxPlayers
7327,Start Player: A Kinda Collectible Card Game,999
7743,"I Don't Know, What Do You Want to Play?",999
15467,Scrimish Card Game,999
2568,The Hammer of Thor: The Game of Norse Mythology,362
12771,Linkee!,200
7322,Pit Fighter: Fantasy Arena,163
15221,Alchemidus,127
9552,Black Powder: Second Edition,120
544,Fairy Meat,100
1977,Rapid Recall,100


En un principio puede parecer que hay errores en el numero total de jugadores, pero algunos juegos pueden admitir grupos muy grandes y el 999 muchas veces significa:

"Prácticamente sin límite de jugadores"

No es un error de captura, sino una forma de codificar una característica especial del juego.

Por ello no vamos a eliminar nada, pero si vamos a añadir una nueva variable para ordenarlos.

In [14]:
def classify_players(x):
    if x <= 2:
        return "2 Players"
    elif x <= 5:
        return "3-5 Players"
    elif x <= 10:
        return "6-10 Players"
    else:
        return "Large Groups"

df_clean["PlayerSegment"] = (
    df_clean["MaxPlayers"]
      .apply(classify_players)
)

In [13]:
df[df["MfgPlaytime"] > 1000][
    ["Name", "MfgPlaytime"]
]

,Name,MfgPlaytime
217,Empires in Arms,12000
243,Advanced Third Reich,2480
285,Russian Front,1200
622,Turning Point: Stalingrad,1800
1117,World in Flames,6000
...,...,...
20618,The Jaws of Victory: Battle of Korsun-Cherkass...,3000
20789,1985: Deadly Northern Lights,5000
21597,Don't Get Got!: Shut Up & Sit Down Special Edi...,1440
21614,"Panzers Last Stand: Battles for Budapest, 1945",1560


Como pasa en el caso anterior, debido a que hay juegos de duraciones indefinidas, como juegos de campaña que dependen de decisiones de los jugadores, tampoco podemos contar como outliers los que marcan una cantidad de horas que se sale de lo comun.
Pot tanto, se concluye que estos valores no son errores de captura sino características inherentes a determinados tipos de juegos (juegos de rol, campañas narrativas o juegos para grupos numerosos). Por ello se decide mantener dichos registros y documentar su impacto potencial en las visualizaciones.

In [15]:
df.to_csv(
    "games_clean.csv",
    index=False
)

In [17]:
from google.colab import files

files.download("games_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>